# 02 — Black–Scholes pricing & Greeks

**Project:** European option pricing, Greeks, and implied volatility.

## Problem
Given spot, strike, time, rates, and volatility, what is a fair European option
price — and how sensitive is it to those inputs?

## Method
Use closed-form Black–Scholes–Merton (`black_scholes_price`, `black_scholes_greeks`)
and recover volatility from a market price via bisection (`implied_volatility`).

## Modules
- `quant_utils.options.black_scholes`


In [ ]:
from datetime import date
from quant_utils.options import (
    OptionContract,
    black_scholes_price,
    black_scholes_greeks,
    implied_volatility,
    year_fraction_to_expiry,
)

spot, strike, r, q, vol = 100.0, 100.0, 0.05, 0.01, 0.20
expiry = date(2026, 12, 18)
as_of = date(2026, 3, 15)
T = year_fraction_to_expiry(expiry, as_of=as_of)
contract = OptionContract("DEMO", "call", strike, expiry)
print(contract)
print(f"T = {T:.4f} years")

call = black_scholes_price(spot, strike, T, r, vol, "call", q)
put = black_scholes_price(spot, strike, T, r, vol, "put", q)
g = black_scholes_greeks(spot, strike, T, r, vol, "call", q)
print(f"call={call:.4f}  put={put:.4f}")
print(g)


In [ ]:
# Round-trip: price → IV → price
iv = implied_volatility(call, spot, strike, T, r, "call", q)
repriced = black_scholes_price(spot, strike, T, r, iv, "call", q)
print(f"true vol={vol:.6f}  implied={iv:.6f}  repriced={repriced:.6f}")


In [ ]:
import matplotlib
matplotlib.use("Agg")  # headless-friendly
import numpy as np
import matplotlib.pyplot as plt

strikes = np.linspace(80, 120, 41)
call_prices = [black_scholes_price(spot, k, T, r, vol, "call", q) for k in strikes]
deltas = [black_scholes_greeks(spot, k, T, r, vol, "call", q).delta for k in strikes]

fig, axes = plt.subplots(1, 2, figsize=(9, 3.5))
axes[0].plot(strikes, call_prices)
axes[0].axvline(spot, color="gray", ls="--", lw=1)
axes[0].set_title("Call price vs strike")
axes[0].set_xlabel("Strike")
axes[1].plot(strikes, deltas)
axes[1].axvline(spot, color="gray", ls="--", lw=1)
axes[1].set_title("Call delta vs strike")
axes[1].set_xlabel("Strike")
for ax in axes:
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## Short result
ATM call/put prices and Greeks match textbook BS identities; implied vol recovers
the input volatility to numerical tolerance on the seeded example.

## Limitations
- European, constant vol / rates — no smiles, jumps, or early exercise.
- IV solver can refuse prices with near-zero vega (deep ITM/OTM, tiny T).
- No dividends schedule, borrow, or settlement quirks.
- Research/education only — not for live market-making or hedging.
